URL: https://www.URL: https://www.emi.ea.govt.nz/Wholesale/Download/DataReport/CSV/UWRCKF

UWRCKF — the report ID for "Scheduled generation outages"
DateFrom / DateTo — date range in YYYYMMDD format

What it pulls: Scheduled generation outages — date, series code (e.g. NZ_H, NZ_T), series name (Hydro/Thermal), and lost megawatts.
Output file: jacques_data/scheduled_outages.csv — creates the folder if it doesn't exist, saves a clean CSV with just the data rows (metadata header stripped out).

In [1]:
#planned outages API

import pandas as pd
import requests
from io import StringIO
from pathlib import Path

DATE_FROM   = "20140101"
DATE_TO     = "20260525"
OUTPUT_PATH = Path("david_data/scheduled_outages.csv")

url = (
    "https://www.emi.ea.govt.nz/Wholesale/Download/DataReport/CSV/UWRCKF"
    f"?DateFrom={DATE_FROM}&DateTo={DATE_TO}"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

lines = response.text.splitlines()
csv_start = next(i for i, l in enumerate(lines) if l.startswith("Timestamp"))
df = pd.read_csv(StringIO("\n".join(lines[csv_start:])))

print(f"✅ {len(df):,} rows × {len(df.columns)} columns")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

✅ 166,817 rows × 4 columns
Saved to david_data/scheduled_outages.csv


In [2]:
# HVDC API

import pandas as pd
import requests
from io import StringIO
from pathlib import Path

DATE_FROM   = "20140101"
DATE_TO     = "20260525"
OUTPUT_PATH = Path("david_data/hvdc_transfer.csv")

url = (
    "https://www.emi.ea.govt.nz/Wholesale/Download/DataReport/CSV/W_HVDC_C"
    f"?DateFrom={DATE_FROM}&DateTo={DATE_TO}"
)

response = requests.get(url, timeout=60)
response.raise_for_status()

lines = response.text.splitlines()
csv_start = next(i for i, l in enumerate(lines) if l.startswith("Period start"))
df = pd.read_csv(StringIO("\n".join(lines[csv_start:])))

print(f"✅ {len(df):,} rows × {len(df.columns)} columns")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

✅ 215,511 rows × 7 columns
Saved to david_data/hvdc_transfer.csv


In [ ]:
#wholesale price

import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import time

OUTPUT_PATH = Path("david_data/wholesale_prices.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://www.emi.ea.govt.nz/Wholesale/Download/DataReport/CSV/W_P_C"

def fetch_year(year: int) -> pd.DataFrame | None:
    url = (
        f"{BASE_URL}?DateFrom={year}0101&DateTo={year}1231"
        "&RegionType=POC_REF"
    )
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        lines = r.text.splitlines()
        csv_start = next((i for i, l in enumerate(lines) if l.startswith("Period start")), None)
        if csv_start is None:
            print(f"  ⚠️  {year}: no data header found")
            return None
        return pd.read_csv(StringIO("\n".join(lines[csv_start:])))
    except Exception as e:
        print(f"  ❌ {year}: {e}")
        return None

frames = []
for year in range(2014, 2027):
    cache = Path(f"david_data/cache_{year}.csv")
    if cache.exists():
        print(f"  📁 {year}: loaded from cache")
        frames.append(pd.read_csv(cache))
        continue

    print(f"  🌐 {year}: fetching...")
    df = fetch_year(year)
    if df is not None:
        df.to_csv(cache, index=False)
        frames.append(df)
        print(f"      {len(df):,} rows")
    time.sleep(1)  # be polite

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ {len(combined):,} rows × {len(combined.columns)} cols → {OUTPUT_PATH}")

  🌐 2014: fetching...
      140,160 rows
  🌐 2015: fetching...
      140,160 rows
  🌐 2016: fetching...
      140,544 rows
  🌐 2017: fetching...
      140,160 rows
  🌐 2018: fetching...
      154,749 rows
  🌐 2019: fetching...
      157,680 rows
  🌐 2020: fetching...
      158,112 rows
  🌐 2021: fetching...
      157,680 rows
  🌐 2022: fetching...
      157,680 rows
  🌐 2023: fetching...
      157,680 rows
  🌐 2024: fetching...
      158,112 rows
  🌐 2025: fetching...
      157,680 rows
  🌐 2026: fetching...
      63,306 rows

✅ 1,883,703 rows × 5 cols → david_data/wholesale_prices.csv


In [7]:
import pandas as pd
from pathlib import Path

files = sorted(Path("david_data").glob("cache_[0-9]*.csv"))
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined.to_csv("david_data/wholesale_prices.csv", index=False)
print(f"✅ {len(combined):,} rows × {len(combined.columns)} cols from {len(files)} files")

✅ 1,883,703 rows × 5 cols from 13 files


In [5]:
#demand by zone

import pandas as pd
import requests
from io import StringIO
from pathlib import Path
import time

OUTPUT_PATH = Path("david_data/demand_by_zone.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://www.emi.ea.govt.nz/Wholesale/Download/DataReport/CSV/W_GD_C"

def fetch_year(year: int) -> pd.DataFrame | None:
    url = (
        f"{BASE_URL}?DateFrom={year}0101&DateTo={year}1231"
    )
    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        lines = r.text.splitlines()
        csv_start = next((i for i, l in enumerate(lines) if l.startswith("Period start")), None)
        if csv_start is None:
            print(f"  ⚠️  {year}: no data header found")
            return None
        return pd.read_csv(StringIO("\n".join(lines[csv_start:])))
    except Exception as e:
        print(f"  ❌ {year}: {e}")
        return None

frames = []
for year in range(2014, 2027):
    cache = Path(f"david_data/cache_demand_{year}.csv")
    if cache.exists():
        print(f"  📁 {year}: loaded from cache")
        frames.append(pd.read_csv(cache))
        continue

    print(f"  🌐 {year}: fetching...")
    df = fetch_year(year)
    if df is not None:
        df.to_csv(cache, index=False)
        frames.append(df)
        print(f"      {len(df):,} rows")
    time.sleep(1)

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(OUTPUT_PATH, index=False)
print(f"\n✅ {len(combined):,} rows × {len(combined.columns)} cols → {OUTPUT_PATH}")

  🌐 2014: fetching...
      87,600 rows
  🌐 2015: fetching...
      87,600 rows
  🌐 2016: fetching...
      87,840 rows
  🌐 2017: fetching...
      87,600 rows
  🌐 2018: fetching...
      87,590 rows
  🌐 2019: fetching...
      87,600 rows
  🌐 2020: fetching...
      87,840 rows
  🌐 2021: fetching...
      87,600 rows
  🌐 2022: fetching...
      87,600 rows
  🌐 2023: fetching...
      87,600 rows
  🌐 2024: fetching...
      87,840 rows
  🌐 2025: fetching...
      87,600 rows
  🌐 2026: fetching...
      35,170 rows

✅ 1,087,080 rows × 6 cols → david_data/demand_by_zone.csv


In [6]:
import pandas as pd
from pathlib import Path

files = sorted(Path("david_data").glob("cache_demand_*.csv"))
combined = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
combined.to_csv("david_data/demand_by_zone.csv", index=False)
print(f"✅ {len(combined):,} rows × {len(combined.columns)} cols from {len(files)} files")

✅ 1,087,080 rows × 6 cols from 13 files
